# 02 - Preprocesamiento de Datos
## Proyecto Final TI24 — Bagging: Random Forests
**Estudiante:** Fabio Mavric

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('✅ Librerías cargadas')

## 1. Carga del Dataset

In [ ]:
df = pd.read_csv('../data/raw/cardio_train.csv', sep=';')

# Convertir edad de días a años
df['age_years'] = (df['age'] / 365).round(1)
df.drop('age', axis=1, inplace=True)

# Eliminar columna ID si existe
if 'id' in df.columns:
    df.drop('id', axis=1, inplace=True)

print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head()

## 2. Limpieza — Eliminación de Outliers

### Criterios médicamente razonables:
- Presión sistólica: entre 60 y 250 mmHg
- Presión diastólica: entre 40 y 200 mmHg
- Altura: entre 120 y 220 cm
- Peso: entre 30 y 200 kg

In [ ]:
rows_before = len(df)

# Filtrar outliers con criterios médicos
df = df[
    (df['ap_hi'] >= 60)  & (df['ap_hi'] <= 250) &
    (df['ap_lo'] >= 40)  & (df['ap_lo'] <= 200) &
    (df['height'] >= 120) & (df['height'] <= 220) &
    (df['weight'] >= 30)  & (df['weight'] <= 200) &
    (df['ap_hi'] >= df['ap_lo'])  # la sistólica debe ser >= diastólica
]

rows_after = len(df)
print(f'Filas antes:  {rows_before:,}')
print(f'Filas después: {rows_after:,}')
print(f'Filas eliminadas: {rows_before - rows_after:,} ({(rows_before-rows_after)/rows_before*100:.2f}%)')

## 3. Feature Engineering

In [ ]:
# Índice de Masa Corporal (IMC)
df['bmi'] = (df['weight'] / (df['height'] / 100) ** 2).round(2)

# Presión de Pulso (diferencial)
df['pulse_pressure'] = df['ap_hi'] - df['ap_lo']

# Grupo de edad
df['age_group'] = pd.cut(df['age_years'],
                          bins=[0, 40, 50, 60, 100],
                          labels=['<40 años', '40-50 años', '50-60 años', '>60 años'])

print('✅ Features creados:')
print('  - bmi: Índice de Masa Corporal')
print('  - pulse_pressure: Presión de pulso')
print('  - age_group: Grupo etario')
df[['bmi', 'pulse_pressure', 'age_group']].describe(include='all')

## 4. Encoding de Variables Categóricas

In [ ]:
# age_group tiene orden → LabelEncoder
le = LabelEncoder()
df['age_group_enc'] = le.fit_transform(df['age_group'].astype(str))

# Variables ya son numéricas (0/1/2/3) → no requieren encoding adicional
# gender, cholesterol, gluc, smoke, alco, active

# Eliminar columna categórica original de age_group
df.drop('age_group', axis=1, inplace=True)

print('Columnas finales del dataset:')
print(df.columns.tolist())
print(f'\nShape: {df.shape}')

## 5. Separación de Features y Target

In [ ]:
X = df.drop('cardio', axis=1)
y = df['cardio']

print(f'Features (X): {X.shape}')
print(f'Target  (y): {y.shape}')
print(f'\nColumnas de X:')
for col in X.columns:
    print(f'  - {col}')

## 6. División Train / Test (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # mantiene proporción de clases
)

print(f'Conjunto de entrenamiento: {X_train.shape[0]:,} filas')
print(f'Conjunto de prueba:        {X_test.shape[0]:,} filas')
print(f'Proporción 80/20: ✅')
print(f'\nBalance en Train — Clase 0: {y_train.value_counts()[0]:,} | Clase 1: {y_train.value_counts()[1]:,}')
print(f'Balance en Test  — Clase 0: {y_test.value_counts()[0]:,} | Clase 1: {y_test.value_counts()[1]:,}')

## 7. Normalización (StandardScaler)

In [ ]:
# Variables numéricas continuas a escalar
scale_cols = ['age_years', 'height', 'weight', 'ap_hi', 'ap_lo', 'bmi', 'pulse_pressure']

scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])

print('✅ Escalado aplicado con StandardScaler')
print('Estadísticas post-escalado (Train):')
X_train[scale_cols].describe().round(3)

## 8. Guardado de Datos Procesados

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print('✅ Archivos guardados en data/processed/')
print('  - X_train.csv')
print('  - X_test.csv')
print('  - y_train.csv')
print('  - y_test.csv')
print('\n✅ Preprocesamiento completado. Continuar con 03_model_main.ipynb')